# FedUCA Extra-A3: K=20 alpha=0.1 + 5-fold CV

Fixes the empty-client crash from Extra-A2 Part 2.
Uses drop_last=True and minimum client size guard.

**Add data:** `ham10000-vit-embeddings-orig` | **GPU T4** | Internet OFF

In [ ]:
import os, json, time, copy, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_extra_a3"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7
EMBED_DIM   = 768

hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    raise FileNotFoundError("Add ham10000-vit-embeddings-orig dataset first.")
EMBED_DIR = os.path.dirname(hits[0])
print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    embs = torch.tensor(d['embeddings'], dtype=torch.float32)
    lbls = torch.tensor(d['labels'],     dtype=torch.long)
    ids  = d['lesion_ids']
    return embs, lbls, ids

emb_train, lbl_train, ids_train = load_split("train")
emb_val,   lbl_val,   ids_val   = load_split("val")
emb_test,  lbl_test,  ids_test  = load_split("test")
emb_tv  = torch.cat([emb_train, emb_val],  dim=0)
lbl_tv  = torch.cat([lbl_train, lbl_val],  dim=0)
ids_tv  = np.concatenate([ids_train, ids_val])

print(f"Train:{emb_train.shape} Val:{emb_val.shape} Test:{emb_test.shape}")
counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = (1.0/counts); weights = (weights/weights.mean()).to(DEVICE)
print(f"Weights: {weights.cpu().numpy().round(3)}")

In [ ]:
# All utilities — with FIXED partition that guarantees min_size samples per client
class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768,4*b),nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,b),  nn.GELU(),nn.LayerNorm(b),  nn.Dropout(0.2),
            nn.Linear(b,4*b),  nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,7))
    def forward(self, x): return self.net(x)

def dirichlet_partition_safe(labels, K, alpha, seed, min_size=32):
    """
    Dirichlet partition with safety guard.
    Any client with fewer than min_size samples gets extra samples
    redistributed from the largest client.
    """
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max())+1):
        ci = np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())

    # Safety: ensure each client has at least min_size samples
    for k in range(K):
        while len(idx[k]) < min_size:
            # Take from largest client
            largest = max(range(K), key=lambda x: len(idx[x]))
            if len(idx[largest]) <= min_size:
                break  # Cannot redistribute
            # Move one sample
            idx[k].append(idx[largest].pop())

    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval(); p, t = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p=torch.cat(p).numpy(); t=torch.cat(t).numpy()
    f1s=[]
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9); f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy":float((p==t).mean()),"macro_f1":float(np.mean(f1s)),
            "f1_per_class":[float(x) for x in f1s]}

def fedavg(states, sizes):
    total=sum(sizes)
    return {k:sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

def local_train(head, embs, labels, lr=5e-3, epochs=5, bs=32):
    if len(embs) == 0:
        return copy.deepcopy(head.state_dict())
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)
    # drop_last=False, use actual batch size capped at dataset size
    actual_bs = min(bs, len(embs))
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=actual_bs, shuffle=True,
                               drop_last=False):
            if len(x) < 2: continue   # skip single-sample batches
            opt.zero_grad(); crit(h(x),y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def compute_prototypes(embs, labels):
    proto=torch.zeros(NUM_CLASSES,EMBED_DIM)
    count=torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask=labels==c
        if mask.sum()>0:
            proto[c]=embs[mask].mean(0); count[c]=float(mask.sum())
    return proto,count

def prototype_guided_aggregation(states, sizes, client_protos, beta=5.0):
    K=len(states); total=sum(sizes)
    g_proto=torch.zeros(NUM_CLASSES,EMBED_DIM)
    g_count=torch.zeros(NUM_CLASSES)
    for k in range(K):
        pk,ck=client_protos[k]
        for c in range(NUM_CLASSES):
            if ck[c]>0: g_proto[c]+=ck[c]*pk[c]; g_count[c]+=ck[c]
    for c in range(NUM_CLASSES):
        if g_count[c]>0: g_proto[c]/=g_count[c]
    cos_d=lambda a,b: 1.0-float((a/(a.norm()+1e-9))@(b/(b.norm()+1e-9)))
    w=torch.zeros(NUM_CLASSES,K)
    for c in range(NUM_CLASSES):
        if g_proto[c].norm()<1e-9:
            for k in range(K): w[c,k]=sizes[k]/total
        else:
            lw=torch.zeros(K)
            for k in range(K):
                pk,ck=client_protos[k]
                lw[k]=-beta*cos_d(pk[c],g_proto[c]) if ck[c]>0 else -1e6
            w[c]=torch.softmax(lw,dim=0)
    fw_key=fb_key=None
    for key in states[0]:
        s=states[0][key].shape
        if len(s)==2 and s[0]==NUM_CLASSES: fw_key=key
        if len(s)==1 and s[0]==NUM_CLASSES: fb_key=key
    avg={}
    for key in states[0]:
        if key==fw_key:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float()
                                      for k in range(K)) for c in range(NUM_CLASSES)])
        elif key==fb_key:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float()
                                      for k in range(K)) for c in range(NUM_CLASSES)])
        else:
            avg[key]=sum((sizes[i]/total)*states[i][key].float() for i in range(K))
    return avg

def run_method(method, K, alpha, seed, n_rounds=60):
    partition=dirichlet_partition_safe(lbl_train, K, alpha, seed)
    c_embs  =[emb_train[idx] for idx in partition]
    c_labels=[lbl_train[idx] for idx in partition]
    sizes   =[max(1,len(idx)) for idx in partition]

    # Print partition stats
    min_s=min(len(idx) for idx in partition)
    max_s=max(len(idx) for idx in partition)
    print(f"    Partition: min={min_s} max={max_s} clients={K}")

    if method=='pga':
        c_protos=[compute_prototypes(c_embs[k],c_labels[k]) for k in range(K)]

    head=BottleneckMLP(ratio=4).to(DEVICE)
    best_val=-1; best_state=None
    t0=time.time()

    for rnd in range(1,n_rounds+1):
        states=[local_train(head,c_embs[k],c_labels[k]) for k in range(K)]
        if method=='pga':
            head.load_state_dict(
                prototype_guided_aggregation(states,sizes,c_protos))
        else:
            head.load_state_dict(fedavg(states,sizes))
        vm=evaluate(head,emb_val.to(DEVICE),lbl_val)
        if vm['macro_f1']>best_val:
            best_val=vm['macro_f1']
            best_state=copy.deepcopy(head.state_dict())
        if rnd%20==0:
            print(f"    r{rnd} val={vm['macro_f1']:.3f} best={best_val:.3f} | {time.time()-t0:.0f}s")

    head.load_state_dict(best_state)
    tm=evaluate(head,emb_test.to(DEVICE),lbl_test)
    return {"method":method,"K":K,"alpha":alpha,"seed":seed,
            "best_val_f1":best_val,"test":tm}

print("All utilities ready with safe partition.")

In [ ]:
# ── PART 1: K=20 at all alpha values ─────────────────────────────────
print("="*60)
print("PART 1: K=20 clients — FedAvg and PGA")
print("="*60)

results_k20=[]
ALPHAS=[1000,1,0.1]; SEEDS=list(range(5))

for method in ['fedavg','pga']:
    for alpha in ALPHAS:
        print(f"\n--- {method} K=20 alpha={alpha} ---")
        for seed in SEEDS:
            print(f"  seed={seed}")
            res=run_method(method,K=20,alpha=alpha,seed=seed)
            results_k20.append(res)
            with open(os.path.join(RESULTS_DIR,
                      f"{method}_k20_a{alpha}_s{seed}.json"),"w") as f:
                json.dump(res,f,indent=2)
            print(f"  {method} K=20 alpha={alpha} seed={seed}: "
                  f"F1={res['test']['macro_f1']:.3f}")
        f1s=[r['test']['macro_f1'] for r in results_k20
             if r['method']==method and r['alpha']==alpha]
        print(f"  >>> {method} K=20 alpha={alpha}: "
              f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR,"k20_results.json"),"w") as f:
    json.dump(results_k20,f,indent=2)
print("\nPart 1 complete.")

In [ ]:
# ── PART 2: 5-fold lesion-grouped cross-validation ───────────────────
print("="*60)
print("PART 2: 5-fold Lesion-Grouped CV (K=4, alpha=1000)")
print("="*60)

results_cv=[]
gkf   =GroupKFold(n_splits=5)
splits=list(gkf.split(np.arange(len(emb_tv)),lbl_tv.numpy(),groups=ids_tv))

for fold,(tr_idx,va_idx) in enumerate(splits):
    print(f"\n--- Fold {fold+1}/5 ---")
    print(f"  train={len(tr_idx)} val={len(va_idx)}")
    fe_tr=emb_tv[tr_idx]; fl_tr=lbl_tv[tr_idx]
    fe_va=emb_tv[va_idx]; fl_va=lbl_tv[va_idx]

    fc=torch.bincount(fl_tr,minlength=NUM_CLASSES).float()
    fw=(1.0/fc); fw=(fw/fw.mean()).to(DEVICE)
    old_w=weights.clone(); weights.data=fw.data

    for method in ['fedavg','pga']:
        partition=dirichlet_partition_safe(fl_tr,4,1000,seed=fold)
        c_embs  =[fe_tr[idx] for idx in partition]
        c_labels=[fl_tr[idx] for idx in partition]
        sizes   =[max(1,len(idx)) for idx in partition]
        if method=='pga':
            c_protos=[compute_prototypes(c_embs[k],c_labels[k])
                      for k in range(4)]
        head=BottleneckMLP(ratio=4).to(DEVICE)
        best_val=-1; best_state=None
        for rnd in range(1,61):
            states=[local_train(head,c_embs[k],c_labels[k])
                    for k in range(4)]
            if method=='pga':
                head.load_state_dict(
                    prototype_guided_aggregation(states,sizes,c_protos))
            else:
                head.load_state_dict(fedavg(states,sizes))
            vm=evaluate(head,fe_va.to(DEVICE),fl_va)
            if vm['macro_f1']>best_val:
                best_val=vm['macro_f1']
                best_state=copy.deepcopy(head.state_dict())
        weights.data=old_w.data
        head.load_state_dict(best_state)
        tm=evaluate(head,emb_test.to(DEVICE),lbl_test)
        res={"method":method,"fold":fold,"val_f1":best_val,"test":tm}
        results_cv.append(res)
        print(f"  {method}: val={best_val:.3f} test={tm['macro_f1']:.3f}")

with open(os.path.join(RESULTS_DIR,"cv_results.json"),"w") as f:
    json.dump(results_cv,f,indent=2)
print("\nPart 2 complete.")

In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────
print("\n"+"="*60)
print("SUMMARY")
print("="*60)

# Scalability table
print("\nScalability (macro-F1, all K values):")
import json as _j, glob as _g
ea_hits=_g.glob("/kaggle/input/**/results_extra_a/*.json",recursive=True)
ea_old=[]
for f in ea_hits:
    d=_j.load(open(f))
    if isinstance(d,list): ea_old.extend(d)
    else: ea_old.append(d)
k10=[r for r in ea_old if r.get('K')==10]

print(f"{'Method':<10} {'K':>4} {'a=1000':>13} {'a=1':>13} {'a=0.1':>13}")
ALPHAS=[1000,1,0.1]
for method in ['fedavg','pga']:
    for K,src in [(10,k10),(20,results_k20)]:
        vals=[]
        for alpha in ALPHAS:
            rows=[r for r in src
                  if r.get('method')==method and r.get('alpha')==alpha]
            if rows:
                f1s=[r['test']['macro_f1'] for r in rows]
                vals.append(f"{np.mean(f1s):.3f}±{np.std(f1s):.3f}")
            else:
                vals.append("n/a")
        label="FedAvg" if method=='fedavg' else "PGA"
        print(f"{label:<10} {K:>4} {vals[0]:>13} {vals[1]:>13} {vals[2]:>13}")

# CV summary
print("\n5-fold CV (K=4, alpha=1000):")
for method in ['fedavg','pga']:
    rows=[r for r in results_cv if r['method']==method]
    label="FedAvg" if method=='fedavg' else "PGA"
    vf1=[r['val_f1'] for r in rows]
    tf1=[r['test']['macro_f1'] for r in rows]
    print(f"  {label}: val={np.mean(vf1):.3f}±{np.std(vf1):.3f} "
          f"test={np.mean(tf1):.3f}±{np.std(tf1):.3f}")
if len([r for r in results_cv if r['method']=='pga'])==5:
    pg=[r['test']['macro_f1'] for r in results_cv if r['method']=='pga']
    fa=[r['test']['macro_f1'] for r in results_cv if r['method']=='fedavg']
    _,pv=ttest_rel(pg,fa)
    print(f"  PGA vs FedAvg CV: p={pv:.4f}")

print(f"\nFiles saved: {RESULTS_DIR}")
print("Save output as dataset: feduca-extra-a3-results")